# 99 — End-to-End MLE Demo

Demostración integral del proyecto de Machine Learning Engineering.

## Objetivos

- Ejecutar el flujo completo desde datos crudos hasta predicciones.
- Utilizar exclusivamente los módulos de `src/`.
- Generar artefactos reproducibles.
- Verificar que el modelo guardado produce los mismos resultados.
- Crear un resumen técnico y de negocio.
- Servir como demostración final del primer curso.


## 1. Flujo general

```text
CSV
 ↓
Carga y validación
 ↓
Separación X / y
 ↓
Preprocesamiento
 ↓
Entrenamiento
 ↓
Evaluación
 ↓
Persistencia
 ↓
Recarga
 ↓
Predicción batch
 ↓
Reportes
```


## 2. Configuración del entorno

En Google Colab:

```python
!git clone <URL_DEL_REPOSITORIO>
%cd customer-intelligence-ml-platform
```


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (candidate / "src").exists():
            PROJECT_ROOT = candidate
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


Project root: /workspaces/customer-intelligence-ml-platform/entregable_D3_16_repositorio_final_curso_1


## 3. Importaciones


In [2]:
import json
import time
import joblib
import numpy as np
import pandas as pd

from src.data import load_customer_data
from src.features import split_features_target
from src.models import (
    load_model,
    predict_customers,
    save_training_artifacts,
    train_model,
)
from src.utils import get_project_path, get_logger

logger = get_logger("notebook.end_to_end_demo")


## 4. Definición de rutas


In [3]:
DATA_PATH = get_project_path(
    "data",
    "raw",
    "customer_churn.csv",
)

MODEL_PATH = get_project_path(
    "artifacts",
    "models",
    "end_to_end_churn_pipeline.joblib",
    create_parent=True,
)

METRICS_PATH = get_project_path(
    "reports",
    "metrics",
    "end_to_end_metrics.json",
    create_parent=True,
)

PREDICTIONS_PATH = get_project_path(
    "reports",
    "predictions",
    "end_to_end_predictions.csv",
    create_parent=True,
)

SUMMARY_PATH = get_project_path(
    "reports",
    "metrics",
    "end_to_end_summary.json",
    create_parent=True,
)

for path in [
    DATA_PATH,
    MODEL_PATH,
    METRICS_PATH,
    PREDICTIONS_PATH,
    SUMMARY_PATH,
]:
    print(path)


/workspaces/customer-intelligence-ml-platform/entregable_D3_16_repositorio_final_curso_1/data/raw/customer_churn.csv
/workspaces/customer-intelligence-ml-platform/entregable_D3_16_repositorio_final_curso_1/artifacts/models/end_to_end_churn_pipeline.joblib
/workspaces/customer-intelligence-ml-platform/entregable_D3_16_repositorio_final_curso_1/reports/metrics/end_to_end_metrics.json
/workspaces/customer-intelligence-ml-platform/entregable_D3_16_repositorio_final_curso_1/reports/predictions/end_to_end_predictions.csv
/workspaces/customer-intelligence-ml-platform/entregable_D3_16_repositorio_final_curso_1/reports/metrics/end_to_end_summary.json


## 5. Carga y validación de datos


In [4]:
start_total = time.perf_counter()

df = load_customer_data(
    DATA_PATH,
    validate=True,
)

logger.info(
    "Dataset loaded with shape %s",
    df.shape,
)

df.head()


2026-10-02 03:26:14,297 | INFO | src.data.data_loader | Loaded customer dataset with shape (1000, 21)
2026-10-02 03:26:14,298 | INFO | notebook.end_to_end_demo | Dataset loaded with shape (1000, 21)


,customer_id,gender,age,region,customer_segment,contract_type,tenure_months,monthly_fee,total_spent,internet_service,...,streaming_service,support_calls,complaints,payment_method,last_payment_delay,digital_usage_score,marketing_score,preferred_contact_hour,internal_campaign_code,churn
0,NT-000001,Masculino,28.0,Lima,Masivo,Anual,40,114.04,3812.50,Fibra,...,No,0,1,Transferencia,2,90.4,69.5,20,CMP-D,0
1,NT-000002,Femenino,50.0,Norte,Masivo,Anual,12,115.97,1468.32,Fibra,...,Sí,1,1,Transferencia,1,63.7,64.2,17,CMP-D,0
2,NT-000003,Masculino,38.0,Lima,Joven digital,Mensual,32,114.42,3128.82,Fibra,...,No,2,0,Tarjeta,5,94.5,63.1,10,CMP-D,1
3,NT-000004,Masculino,34.0,Norte,Joven digital,Anual,54,41.89,2037.76,Sin internet,...,No,1,1,Tarjeta,5,33.3,53.0,12,CMP-C,0
4,NT-000005,Femenino,37.0,Lima,Joven digital,Bianual,9,91.54,790.10,Fibra,...,No,2,1,Tarjeta,2,73.8,85.7,12,CMP-B,0


In [5]:
data_summary = {
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "missing_values": int(df.isna().sum().sum()),
    "duplicate_ids": int(
        df["customer_id"].duplicated().sum()
    ),
    "churn_rate": float(
        df["churn"].mean()
    ),
}

data_summary


{'rows': 1000,
 'columns': 21,
 'missing_values': 256,
 'duplicate_ids': 0,
 'churn_rate': 0.24}

## 6. Separación de variables


In [6]:
X, y = split_features_target(df)

print(f"Feature matrix: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target mean: {y.mean():.2%}")


Feature matrix: (1000, 18)
Target shape: (1000,)
Target mean: 24.00%


## 7. Entrenamiento


In [7]:
training_start = time.perf_counter()

training_result = train_model(
    df,
    test_size=0.20,
    random_state=42,
)

training_seconds = (
    time.perf_counter()
    - training_start
)

training_result.metrics


{'accuracy': 0.645,
 'precision': 0.37894736842105264,
 'recall': 0.75,
 'f1': 0.5034965034965035,
 'roc_auc': 0.728344298245614}

In [8]:
print(
    f"Train rows: {training_result.train_rows:,}"
)
print(
    f"Test rows: {training_result.test_rows:,}"
)
print(
    f"Training seconds: {training_seconds:.4f}"
)


Train rows: 800
Test rows: 200
Training seconds: 0.2731


## 8. Guardado de artefactos


In [9]:
save_training_artifacts(
    training_result,
    model_path=MODEL_PATH,
    metrics_path=METRICS_PATH,
)

print(f"Model saved: {MODEL_PATH.exists()}")
print(f"Metrics saved: {METRICS_PATH.exists()}")


Model saved: True
Metrics saved: True


## 9. Recarga del modelo


In [10]:
loaded_pipeline = load_model(
    MODEL_PATH
)

print(type(loaded_pipeline))
print(
    loaded_pipeline.named_steps.keys()
)


<class 'sklearn.pipeline.Pipeline'>
dict_keys(['preprocessor', 'classifier'])


## 10. Comparación antes y después de serializar


In [11]:
sample_features = X.head(25)

original_probabilities = (
    training_result.pipeline.predict_proba(
        sample_features
    )[:, 1]
)

loaded_probabilities = (
    loaded_pipeline.predict_proba(
        sample_features
    )[:, 1]
)

serialization_difference = np.abs(
    original_probabilities
    - loaded_probabilities
)

pd.Series(
    serialization_difference
).describe()


count    25.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
dtype: float64

In [12]:
serialization_consistent = np.allclose(
    original_probabilities,
    loaded_probabilities,
)

serialization_consistent


True

## 11. Preparación del dataset de inferencia


In [13]:
inference_df = df.drop(
    columns=["churn"]
).copy()

print(inference_df.shape)
inference_df.head()


(1000, 20)


,customer_id,gender,age,region,customer_segment,contract_type,tenure_months,monthly_fee,total_spent,internet_service,tv_service,streaming_service,support_calls,complaints,payment_method,last_payment_delay,digital_usage_score,marketing_score,preferred_contact_hour,internal_campaign_code
0,NT-000001,Masculino,28.0,Lima,Masivo,Anual,40,114.04,3812.50,Fibra,Sí,No,0,1,Transferencia,2,90.4,69.5,20,CMP-D
1,NT-000002,Femenino,50.0,Norte,Masivo,Anual,12,115.97,1468.32,Fibra,No,Sí,1,1,Transferencia,1,63.7,64.2,17,CMP-D
2,NT-000003,Masculino,38.0,Lima,Joven digital,Mensual,32,114.42,3128.82,Fibra,Sí,No,2,0,Tarjeta,5,94.5,63.1,10,CMP-D
3,NT-000004,Masculino,34.0,Norte,Joven digital,Anual,54,41.89,2037.76,Sin internet,No,No,1,1,Tarjeta,5,33.3,53.0,12,CMP-C
4,NT-000005,Femenino,37.0,Lima,Joven digital,Bianual,9,91.54,790.10,Fibra,No,No,2,1,Tarjeta,2,73.8,85.7,12,CMP-B


## 12. Predicción batch


In [14]:
prediction_start = time.perf_counter()

predictions = predict_customers(
    loaded_pipeline,
    inference_df,
    threshold=0.50,
)

prediction_seconds = (
    time.perf_counter()
    - prediction_start
)

predictions.head()


,customer_id,churn_probability,churn_prediction
0,NT-000001,0.229313,0
1,NT-000002,0.496753,0
2,NT-000003,0.467431,0
3,NT-000004,0.388501,0
4,NT-000005,0.267129,0


In [15]:
print(
    f"Predicted rows: {len(predictions):,}"
)
print(
    f"Prediction seconds: {prediction_seconds:.4f}"
)
print(
    "Rows per second: "
    f"{len(predictions) / prediction_seconds:,.2f}"
)


Predicted rows: 1,000
Prediction seconds: 0.0271
Rows per second: 36,940.86


## 13. Enriquecimiento de resultados


In [16]:
business_context = inference_df[
    [
        "customer_id",
        "region",
        "customer_segment",
        "contract_type",
        "monthly_fee",
        "tenure_months",
        "support_calls",
        "complaints",
        "last_payment_delay",
    ]
].copy()

final_predictions = (
    business_context
    .merge(
        predictions,
        on="customer_id",
        how="left",
        validate="one_to_one",
    )
)

final_predictions.head()


,customer_id,region,customer_segment,contract_type,monthly_fee,tenure_months,support_calls,complaints,last_payment_delay,churn_probability,churn_prediction
0,NT-000001,Lima,Masivo,Anual,114.04,40,0,1,2,0.229313,0
1,NT-000002,Norte,Masivo,Anual,115.97,12,1,1,1,0.496753,0
2,NT-000003,Lima,Joven digital,Mensual,114.42,32,2,0,5,0.467431,0
3,NT-000004,Norte,Joven digital,Anual,41.89,54,1,1,5,0.388501,0
4,NT-000005,Lima,Joven digital,Bianual,91.54,9,2,1,2,0.267129,0


## 14. Bandas de riesgo


In [17]:
final_predictions["risk_band"] = pd.cut(
    final_predictions[
        "churn_probability"
    ],
    bins=[
        -np.inf,
        0.35,
        0.55,
        0.75,
        np.inf,
    ],
    labels=[
        "Bajo",
        "Medio",
        "Alto",
        "Muy alto",
    ],
)

final_predictions[
    "risk_band"
].value_counts()


risk_band
Bajo        370
Medio       291
Alto        226
Muy alto    113
Name: count, dtype: int64

## 15. Priorización para retención


In [18]:
final_predictions[
    "priority_score"
] = (
    final_predictions[
        "churn_probability"
    ]
    * final_predictions[
        "monthly_fee"
    ]
)

priority_list = (
    final_predictions
    .sort_values(
        "priority_score",
        ascending=False,
    )
)

priority_list.head(20)


,customer_id,region,customer_segment,contract_type,monthly_fee,tenure_months,support_calls,complaints,last_payment_delay,churn_probability,churn_prediction,risk_band,priority_score
959,NT-000960,Centro,Familia,Mensual,379.72,35,24,1,87,0.521791,1,Medio,198.134319
765,NT-000766,Norte,Masivo,Mensual,204.85,15,23,0,76,0.882378,1,Muy alto,180.755100
84,NT-000085,Lima,Joven digital,Mensual,218.56,23,14,0,75,0.735245,1,Alto,160.695069
921,NT-000922,Sur,Joven digital,Mensual,226.26,5,15,0,46,0.673376,1,Alto,152.358077
968,NT-000969,Norte,Premium,Mensual,156.88,20,2,3,3,0.908962,1,Muy alto,142.597892
162,NT-000163,Lima,Familia,Anual,201.87,12,24,0,78,0.705340,1,Alto,142.386984
707,NT-000708,Norte,Joven digital,Mensual,383.41,51,23,0,64,0.362450,0,Medio,138.967030
834,NT-000835,Lima,Familia,Anual,282.15,8,19,0,77,0.464217,0,Medio,130.978811
567,NT-000568,Oriente,Premium,Mensual,156.97,6,1,0,6,0.776255,1,Muy alto,121.848793
998,NT-000999,Lima,Premium,Mensual,160.70,33,1,1,4,0.745388,1,Alto,119.783848


## 16. Resumen de negocio


In [19]:
business_summary = {
    "total_customers": int(
        len(final_predictions)
    ),
    "predicted_churn_customers": int(
        final_predictions[
            "churn_prediction"
        ].sum()
    ),
    "predicted_churn_rate": float(
        final_predictions[
            "churn_prediction"
        ].mean()
    ),
    "average_probability": float(
        final_predictions[
            "churn_probability"
        ].mean()
    ),
    "monthly_fee_at_risk": float(
        final_predictions.loc[
            final_predictions[
                "churn_prediction"
            ] == 1,
            "monthly_fee",
        ].sum()
    ),
    "very_high_risk_customers": int(
        (
            final_predictions[
                "risk_band"
            ] == "Muy alto"
        ).sum()
    ),
}

business_summary


{'total_customers': 1000,
 'predicted_churn_customers': 426,
 'predicted_churn_rate': 0.426,
 'average_probability': 0.4507544962571528,
 'monthly_fee_at_risk': 42710.32,
 'very_high_risk_customers': 113}

## 17. Validaciones end-to-end


In [20]:
validation_checks = {
    "model_file_exists": MODEL_PATH.exists(),
    "metrics_file_exists": METRICS_PATH.exists(),
    "serialization_consistent": bool(
        serialization_consistent
    ),
    "prediction_row_count_matches": (
        len(predictions)
        == len(inference_df)
    ),
    "probabilities_in_range": bool(
        predictions[
            "churn_probability"
        ].between(0, 1).all()
    ),
    "predictions_are_binary": bool(
        set(
            predictions[
                "churn_prediction"
            ].unique()
        ).issubset({0, 1})
    ),
    "customer_ids_unique": bool(
        predictions[
            "customer_id"
        ].is_unique
    ),
}

validation_checks


{'model_file_exists': True,
 'metrics_file_exists': True,
 'serialization_consistent': True,
 'prediction_row_count_matches': True,
 'probabilities_in_range': True,
 'predictions_are_binary': True,
 'customer_ids_unique': True}

In [21]:
if not all(
    validation_checks.values()
):
    raise RuntimeError(
        "End-to-end validation failed."
    )

print("All end-to-end checks passed.")


All end-to-end checks passed.


## 18. Exportación de predicciones


In [22]:
priority_list.to_csv(
    PREDICTIONS_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(PREDICTIONS_PATH)


/workspaces/customer-intelligence-ml-platform/entregable_D3_16_repositorio_final_curso_1/reports/predictions/end_to_end_predictions.csv


## 19. Resumen técnico


In [23]:
total_seconds = (
    time.perf_counter()
    - start_total
)

technical_summary = {
    "training_seconds": float(
        training_seconds
    ),
    "prediction_seconds": float(
        prediction_seconds
    ),
    "total_seconds": float(
        total_seconds
    ),
    "prediction_rows_per_second": float(
        len(predictions)
        / prediction_seconds
    ),
    "model_class": type(
        loaded_pipeline.named_steps[
            "classifier"
        ]
    ).__name__,
    "pipeline_steps": list(
        loaded_pipeline.named_steps.keys()
    ),
    "metrics": training_result.metrics,
}

technical_summary


{'training_seconds': 0.2730617180000081,
 'prediction_seconds': 0.027070294999703037,
 'total_seconds': 1.2583265820003362,
 'prediction_rows_per_second': 36940.86082220272,
 'model_class': 'LogisticRegression',
 'pipeline_steps': ['preprocessor', 'classifier'],
 'metrics': {'accuracy': 0.645,
  'precision': 0.37894736842105264,
  'recall': 0.75,
  'f1': 0.5034965034965035,
  'roc_auc': 0.728344298245614}}

## 20. Exportación del resumen integral


In [24]:
end_to_end_summary = {
    "data": data_summary,
    "business": business_summary,
    "technical": technical_summary,
    "validation": validation_checks,
    "artifacts": {
        "model": str(MODEL_PATH),
        "metrics": str(METRICS_PATH),
        "predictions": str(
            PREDICTIONS_PATH
        ),
    },
}

SUMMARY_PATH.write_text(
    json.dumps(
        end_to_end_summary,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(SUMMARY_PATH)


/workspaces/customer-intelligence-ml-platform/entregable_D3_16_repositorio_final_curso_1/reports/metrics/end_to_end_summary.json


## 21. Verificación de archivos


In [25]:
generated_files = pd.DataFrame({
    "artifact": [
        "model",
        "metrics",
        "predictions",
        "summary",
    ],
    "path": [
        MODEL_PATH,
        METRICS_PATH,
        PREDICTIONS_PATH,
        SUMMARY_PATH,
    ],
})

generated_files["exists"] = (
    generated_files[
        "path"
    ].apply(
        lambda path: Path(path).exists()
    )
)

generated_files["size_bytes"] = (
    generated_files[
        "path"
    ].apply(
        lambda path: (
            Path(path).stat().st_size
            if Path(path).exists()
            else 0
        )
    )
)

generated_files


,artifact,path,exists,size_bytes
0,model,/workspaces/customer-intelligence-ml-platform/...,True,7002
1,metrics,/workspaces/customer-intelligence-ml-platform/...,True,207
2,predictions,/workspaces/customer-intelligence-ml-platform/...,True,92706
3,summary,/workspaces/customer-intelligence-ml-platform/...,True,1623


## 22. Lectura de artefactos


In [26]:
metrics_payload = json.loads(
    METRICS_PATH.read_text(
        encoding="utf-8"
    )
)

summary_payload = json.loads(
    SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

print(metrics_payload.keys())
print(summary_payload.keys())


dict_keys(['metrics', 'train_rows', 'test_rows'])
dict_keys(['data', 'business', 'technical', 'validation', 'artifacts'])


## 23. Ejecución mediante CLI


In [27]:
import subprocess

cli_process = subprocess.run(
    [
        sys.executable,
        "main.py",
        "train",
        "--data",
        str(DATA_PATH),
        "--model-output",
        str(
            get_project_path(
                "artifacts",
                "models",
                "cli_churn_pipeline.joblib",
                create_parent=True,
            )
        ),
        "--metrics-output",
        str(
            get_project_path(
                "reports",
                "metrics",
                "cli_training_metrics.json",
                create_parent=True,
            )
        ),
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

print(cli_process.stdout)

if cli_process.returncode != 0:
    print(cli_process.stderr)


{'accuracy': 0.645, 'precision': 0.37894736842105264, 'recall': 0.75, 'f1': 0.5034965034965035, 'roc_auc': 0.728344298245614}



## 24. Ejecución de pruebas


In [28]:
test_process = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

print(test_process.stdout)

if test_process.returncode != 0:
    print(test_process.stderr)


.........                                                                [100%]
9 passed in 1.24s



## 25. Resultado final


In [29]:
project_ready = (
    all(
        validation_checks.values()
    )
    and cli_process.returncode == 0
    and test_process.returncode == 0
)

print(
    "PROJECT READY FOR NEXT COURSE"
    if project_ready
    else "PROJECT REQUIRES REVIEW"
)


PROJECT READY FOR NEXT COURSE


## 26. Preguntas para estudiantes

1. ¿Qué componentes se guardan dentro del pipeline?
2. ¿Por qué se compara el modelo antes y después de serializar?
3. ¿Qué diferencia existe entre una métrica técnica y una métrica de negocio?
4. ¿Qué artefactos deberían versionarse con Git y cuáles con DVC?
5. ¿Qué parte del flujo será utilizada por FastAPI?
6. ¿Qué validaciones deberían ejecutarse automáticamente en CI/CD?
7. ¿Qué cambiaría para ejecutar este proceso diariamente?


## 27. Checklist de cierre

- [ ] Se cargaron y validaron los datos.
- [ ] Se entrenó el pipeline.
- [ ] Se calcularon métricas.
- [ ] Se guardó el modelo.
- [ ] Se recargó el artefacto.
- [ ] Se verificó consistencia.
- [ ] Se generaron predicciones batch.
- [ ] Se crearon bandas de riesgo.
- [ ] Se priorizaron clientes.
- [ ] Se exportaron resultados.
- [ ] Se generó el resumen integral.
- [ ] Se ejecutó el CLI.
- [ ] Se ejecutaron pruebas.
- [ ] El proyecto fue aprobado.


## Resultado esperado

Este notebook demuestra que el proyecto puede ejecutarse de extremo a extremo
sin depender de lógica manual dispersa en celdas.

El mismo pipeline será reutilizado posteriormente por:

- FastAPI;
- Docker;
- AWS;
- Prometheus;
- procesos de reentrenamiento.
